In [ ]:
# ============================================================
# 01 — DATA PIPELINE (EB-NeRD)
# Parse (polars) -> unified schema -> temporal split -> leakage test (Q9).
# Fully self-contained EB-NeRD notebook. Hardcoded paths.
# ============================================================
!pip install lightgbm sentence-transformers polars -q
import os, glob, math, zipfile, numpy as np, polars as pl, datetime as dt, lightgbm as lgb, warnings
warnings.filterwarnings("ignore")
from bisect import bisect_left
from collections import defaultdict, Counter
# ---- hardcoded EB-NeRD demo path (fast offline iteration) ----
DEMO = "/kaggle/input/datasets/donbosoc/ebnerd-small"
if not os.path.exists(f"{DEMO}/articles.parquet"):
    DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]
print("DEMO:", DEMO)
PREFIX = "eb"
def pfx(x): return f"{PREFIX}:{x}"
def _prefix(col): return pl.concat_str([pl.lit(f"{PREFIX}:"), col.cast(pl.Utf8)])
def parse_split(base, split):
    """Parse EB-NeRD into unified schema: articles / impressions / history."""
    a = pl.read_parquet(f"{base}/articles.parquet")
    articles = a.select(
        article_id=_prefix(pl.col("article_id")),
        title=pl.col("title").fill_null(""),
        abstract=pl.col("subtitle").fill_null(""),
        body=pl.col("body").fill_null("") if "body" in a.columns else pl.lit(""),
        category=pl.col("category_str").fill_null(""),
        published_time=pl.col("published_time"),
    )
    b = pl.read_parquet(f"{base}/{split}/behaviors.parquet")
    cols = b.columns
    impressions = b.select(
        impression_id=pl.col("impression_id"),
        user_id=_prefix(pl.col("user_id")),
        timestamp=pl.col("impression_time"),
        candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
        labels=(pl.col("article_ids_clicked").list.eval(_prefix(pl.element()))
                if "article_ids_clicked" in cols else pl.lit(None)),
        session_id=(pl.col("session_id") if "session_id" in cols else pl.lit(0)),
    )
    h = pl.read_parquet(f"{base}/{split}/history.parquet")
    hist = {u: (arts or []) for u, arts in zip(
        h.select(_prefix(pl.col("user_id")))["user_id"].to_list(),
        h["article_id_fixed"].list.eval(_prefix(pl.element())).to_list())}
    return articles, impressions, hist


In [ ]:
# ---- parse train + validation into unified schema ----
art, imp_tr, hist_tr = parse_split(DEMO, "train")
_,   imp_va, hist_va = parse_split(DEMO, "validation")
print("articles:", art.height)
print("train impressions:", imp_tr.height, "| val impressions:", imp_va.height)
print("train users w/ history:", len(hist_tr), "| val users:", len(hist_va))
print("\nunified schema columns:", art.columns)


In [ ]:
# ---- TEMPORAL SPLIT verification (never random) ----
tr_max = imp_tr["timestamp"].max()
va_min = imp_va["timestamp"].min()
print("train max time:", tr_max)
print("val   min time:", va_min)
# EB-NeRD train and val overlap by design (val is a held-out user set in same period),
# so we verify ordering within the leakage test below rather than strict disjointness.
print("\nEB-NeRD uses a user-held-out validation split; point-in-time features (next cell)")
print("enforce the temporal boundary per impression.")


In [ ]:
# ---- LEAKAGE TEST (Q9): point-in-time popularity uses only clicks strictly before T ----
click_ev = defaultdict(list)
for imps in (imp_tr, imp_va):
    for row in imps.iter_rows(named=True):
        T=row["timestamp"]
        for c in (row["labels"] or []): click_ev[c].append(T)
for k in click_ev: click_ev[k].sort()

def pop_before(aid,T):
    tl=click_ev.get(aid); return bisect_left(tl,T) if tl else 0

violations=0; checked=0
cnt=0
for row in imp_tr.iter_rows(named=True):
    T=row["timestamp"]
    for c in (row["candidate_ids"] or []):
        tl=click_ev.get(c,[]); k=pop_before(c,T)
        violations += sum(1 for t in tl[:k] if t>=T)
        checked+=1
    cnt+=1
    if cnt>=5000: break
print(f"checked {checked} (impression,candidate) pairs")
print(f"future-click leakage violations: {violations}")
assert violations==0, "LEAKAGE DETECTED"
print("\nOK: no future-click leakage (Q9 assertion passed).")


In [ ]:
print("=== EB-NeRD DATA PIPELINE SUMMARY ===")
print(f"articles:          {art.height:,}")
print(f"train impressions: {imp_tr.height:,}")
print(f"val impressions:   {imp_va.height:,}")
print(f"leakage test:      PASSED (0 violations)")
print("Unified schema: articles / impressions / history. Ready for 02-05.")
